# TP N°1 ACN

Propon ́e de manera precisa una regla de comportamiento de pasajeros
Pens ́a en tu experiencia o busc ́a videos y responde: cu ́anto tarda en levantarse y dejar
lugar un pasajero sentado en el pasillo para que pase un pasajero que va a ventana?
Si una persona llega a su fila y su asiento est ́a vac ́ıo, cu ́anto tarde en sentarse? y si
tiene carry on?

Desde el momento en que un pasajero sube el avión, tiene un repertorio limitado de acciones:
- Avanzar.
- Levantarse.
- Guardar equipaje.
- Sentarse.

Asimismo, pueden adoptar algunos de los siguientes estados:
- Posición actual.
- Asiento.
- Asiento conseguido.
- Tiene carry-on (a mano).
- Sentado.
- Pasajero delante (en pasillo central).
- Pasajero en pasillo (ya estando sentado en fila junto al pasillo).

Vamos a asumir las siguientes reglas de comportamiento para los pasajeros:
- El pasajero que llega a su asiento vacío y no posee carry-on se demora de 4 a 12 segundos en sentarse.
- El pasajero que llega a su asiento vacío y que sí posee carry-on se demora de 12 a 32 segundos en sentarse.
- Cada pasajero que debe levantarse para habilitar la llegada al asiento agrega de 3 a 5 segundos al tiempo total de onboarding.
- Si se requiere guardar el carry-on y esperar para sentarse, necesariamente se debe guardar el carry-on primero para luego esperar a que se le deje pasar.



In [ ]:
# Definimos agentes y modelo
import random
import numpy as np

STORE_CARRYON_MIN = 12
STORE_CARRYON_MAX = 32

SIT_MIN = 4 
SIT_MAX = 12

GET_UP_MIN = 3
GET_UP_MAX = 5

class Plane:

    '''
    Para seleccionar el método de llenado, se tienen cuatro opciones:
    - 'btf' para la política Back-to-Front.
    - 'rand' para la política aleatoria.
    - 'wilma' para la política Window-Middle-Aisle.
    - 'stfn' para el método de Steffen.
    '''

    def __init__(self, n=100, p=0.5, onboarding_method='rand'):
        self.queue:list[tuple[int,int]] = list()

        # Espacio físico
        self.plane_grid:np.ndarray = np.zeros((25, 5))

        # Reloj global
        self.time:int = 0

        # Asiento-pasajero
        self.passenger_at_seat:dict[tuple[int,int], Passenger] = dict()

        # Generemos la queue
        if onboarding_method == 'btf' or onboarding_method == 'rand':

            for row in range(25):
                for col in range(5):
                    if (col != 2): 
                        self.queue.append((row,col))

            random.shuffle(self.queue)

            if onboarding_method == 'btf':
                self.queue.sort(key=lambda tup: tup[0], reverse=True)

        elif onboarding_method == 'wilma':

            for row in range(25):
                for col in range(5):
                    if (col != 2): 
                        self.queue.append((row,col))
            
            random.shuffle(self.queue)
            self.queue.sort(key=lambda tup: tup[1] % 2 == 0, reverse=True)
                        
        elif onboarding_method == 'stfn':

            for col in [0,4]:
                for row in range(24,-1,-2):
                    self.queue.append((row,col))
            
            for col in [0,4]:
                for row in range(23,0,-2):
                    self.queue.append((row,col))

            for col in [1,3]:
                for row in range(24,-1,-2):
                    self.queue.append((row,col))
            
            for col in [1,3]:
                for row in range(23,0,-2):
                    self.queue.append((row,col))
        
        else:
            raise ValueError("Método de embarque inválido:", onboarding_method)

        # Creamos lista de agentes
        self.agents:list[Passenger] = list()
        for seat in self.queue:
            carryon = bool(random.choices([0,1], weights=[p, 1-p])[0])
            pasajero = Passenger(seat=seat, carryon=carryon, plane=self)
            self.agents.append(pasajero) # type: ignore
            self.passenger_at_seat[seat] = pasajero

    def is_cell_occupied(self, cell:tuple[int,int]):
        if self.plane_grid[cell] == 1:
            return True
        return False

    def step(self):
        for agent in self.agents:
            agent.activate()
        for agent in self.agents:
            agent.walk_forward()
        self.time += 1

class Passenger:
    def __init__(self, seat:tuple[int,int], carryon:bool, plane:Plane): # type: ignore
        self.seat:tuple[int,int] = seat
        self.carryon:bool = carryon
        self.arrived_at_seat:bool = False
        self.is_active = False
        self.cell: tuple[int, int] | None = None
        self.plane = plane

        self.state:str = 'walking' # Estados: 'walking', 'storing_carryon', 'sitting', 'exchanging_positions'
        self.timer:int = 0  # segundos restantes por cada acción
        if carryon is True:
            self.carryon_stored:bool | None = False
        else:
            self.carryon_stored:bool | None = None
        # quizás si ya guardo carryon
        # quizás si se esta levantando
        # quizás tiempo parado

    def activate(self):
        if self.is_active:
            return
        self.is_active = True
        print(f"Agente del asiento {self.seat} ingresa al avión en el momento t={self.plane.time} segundos.")

    def orchestrator(self):
        # Si el pasajero aún no esta activo entonces no tiene acción por concretar
        if not self.is_active:
            return
        # Si ya llego a su asiento, no hace nada, se levantara cuando alguien le diga
        if self.arrived_at_seat:
            return
        if self.state == 'storing_carryon':
            self.store_carryon()
        elif self.state == 'sitting':
            self.sit_down()
        else:
            self.walk_forward()
    # def is_someone_over_there(self):
    #     plane_hall_view = (sel)
    #     if self.plane.plane_grid[self]
    def store_carryon(self):
        self.timer -= 1
        if self.timer == 0:
            self.carryon_stored = True
            self.state = 'walking'
    def sit_down(self):
        self.timer -= 1
        if self.timer == 0:
            self.arrived_at_seat = True
            print(f"Agente del asiento", self.seat, "se sienta en el momento t={self.plane.time} segundos.")
    def get_up(self):
        if self.state != 'exchanging_positions':
            self.state = 'exchanging_positions'

    def walk_forward(self):
        if self.cell == None:
            next_cell = (0,2)
        else:
            cell_row, cell_col = self.cell
            seat_row, seat_col = self.seat
            if cell_row < seat_row:
                next_cell = (cell_row + 1, cell_col)
            elif cell_col == 2 and self.carryon and not self.carryon_stored:
                # Llega al pasillo de la fila, antes de ir al asiento guarda el equipaje y luego se asietno o espera que lo dejen pasar 
                self.state = 'storing_carryon'
                self.timer = random.randint(SIT_MIN, SIT_MAX)
                print(f"Agente del asiento", self.seat, "guarda su equipaje de mano en el momento t={self.plane.time} segundos.")
                return
            elif cell_col < seat_col:
                next_cell = (cell_row, cell_col + 1)
            elif cell_col > seat_col:
                next_cell = (cell_row, cell_col - 1)
            else:
                self.state = 'sitting'
                self.timer = random.randint(SIT_MIN, SIT_MAX)
                return
        if self.plane.is_cell_occupied(next_cell): # type: ignore}
            cell_row, cell_col = self.cell # type: ignore
            next_row, next_col = next_cell
            seat_row, seat_col = self.seat
            if cell_row == next_row and next_col != cell_col:
                adjacent_passenger = self.plane.passenger_at_seat(next_cell) # type: ignore
                if adjacent_passenger.arrived_at_seat and adjacent_passenger.state != 'exchanging_positions':
                    adjacent_passenger.get_up()
                return
            # Luego el otro caso, es columna 2 en el pasillo alguien delante, ya esta
            return
        prev_cell = self.cell
        self.cell = next_cell # type: ignore
        self.plane.plane_grid[prev_cell] = 0
        self.plane.plane_grid[self.cell] = 1
        print("Agente del asiento", self.seat, " avanza a ", self.cell)
        if self.cell == self.seat:
            self.arrived_at_seat = True